# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.


In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

# # Adjust this path if your repo is stored elsewhere in Drive.
# PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"
PROJECT_ROOT = "D:/hku/COMP 4329/Assignment1_2026"

In [ ]:
# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en

---

## Section 0 — Environment Setup

Mount Google Drive and install dependencies.


In [ ]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

---

## Section 1 — Download Data _(delete before submitting)_

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.


In [ ]:
from Tools.download import download_mini

download_mini(data_dir="_data")

---

## Section 2 — Preprocess Data _(delete before submitting)_

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.


In [3]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

Generating train examples…


100%|██████████| 150/150 [00:06<00:00, 22.76it/s]


  30293 questions in total
Generating dev examples…


100%|██████████| 48/48 [00:02<00:00, 16.64it/s]


  10570 questions in total
Generating word embedding…


...

100%|██████████| 30293/30293 [00:12<00:00, 2352.99it/s]


  Built 30169 / 30293 instances
Processing dev examples…


100%|██████████| 10570/10570 [00:05<00:00, 2061.63it/s]


  Built 10465 / 10570 instances
Saving word embedding…
Saving char embedding…
Saving train eval…
Saving dev eval…
Saving word dictionary…
Saving char dictionary…
Saving dev meta…

Preprocessing complete.
  Outputs → _data/


{'train_record_file': '_data\\train.npz',
 'dev_record_file': '_data\\dev.npz',
 'word_emb_file': '_data\\word_emb.json',
 'char_emb_file': '_data\\char_emb.json',
 'train_eval_file': '_data\\train_eval.json',
 'dev_eval_file': '_data\\dev_eval.json',
 'word2idx_file': '_data\\word2idx.json',
 'char2idx_file': '_data\\char2idx.json',
 'dev_meta_file': '_data\\dev_meta.json'}

---

## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.


In [2]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    num_steps  = 60000,
    batch_size = 8,
    seed       = 42,

    # ── vanilla recipe: SGD, no scheduler, NLL loss ───────────────────────
    optimizer_name = "sgd",
    scheduler_name = "none",
    loss_name      = "qa_nll",
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

RuntimeError: The size of tensor a (400) must match the size of tensor b (96) at non-singleton dimension 1

---

## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.


In [ ]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")